# Creating model

In [1]:
from app.model.model import TwoTowerModel, PlaceTower, UserTower
from app.model.contrastive_loss import ContrastiveLoss
from app.model.TwoTowerDataset import TwoTowerDataset, collate_fn, pad_cuisine, UserEmbeddingDataset, PlaceEmbeddingDataset, user_collate_fn, place_collate_fn
import torch
import torch.nn as nn


In [2]:
# Helper to find max ID + 1 for Embedding layers
def get_vocab_size(dataset, tower_key, feature_key):
    max_id = 0
    for i in range(len(dataset)):
        val = dataset.data[i] # Access raw row
        # Note: adjust index based on SQL position if needed, 
        # but easier to use the dataset's __getitem__ keys:
        item = dataset[i][tower_key][feature_key]
        if isinstance(item, list):
            if item: max_id = max(max_id, max(item))
        else:
            max_id = max(max_id, item)
    return int(max_id + 1)

In [3]:
from torch.utils.data import DataLoader
import psycopg2
from torch.utils.data import random_split
conn = psycopg2.connect(database='reco_db', user='reco_user',password='reco_pass')
dataset = TwoTowerDataset(conn)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # reproducible
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)




In [4]:
# for i, batch in enumerate(loader):
#     print("Got batch", i)
#     break

In [5]:
# loading num_cuisines from data
user_cuisines = set()
place_cuisines = set()

for i in range(len(dataset)):
    item = dataset[i]
    
    # Handle User Cuisines
    u_c = item['user']['cuisine_ids']
    if isinstance(u_c, list):
        user_cuisines.update(u_c)
    else:
        user_cuisines.add(u_c)
        
    # Handle Place Cuisines
    p_c = item['place']['cuisine_ids']
    if isinstance(p_c, list):
        place_cuisines.update(p_c)
    else:
        place_cuisines.add(p_c)

NUM_USERCUISINES = len(user_cuisines)
NUM_CUISINES = len(place_cuisines)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# load num_cuisines from data
BERT_DIM = 1536 
HIDDEN_DIM = 128

model_config = {
    "user_tower": {
        "numeric_dim": 3,
        "ordinal_dim": 3,
        "cuisine_vocab_size": NUM_USERCUISINES,
        "bert_dim": BERT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dress_vocab": get_vocab_size(dataset, "user", "dress_preference_id"),
        "activity_vocab": get_vocab_size(dataset, "user", "activity_id"),
        "ambience_vocab": get_vocab_size(dataset, "user", "ambience_id"),
        "hijos_vocab": get_vocab_size(dataset, "user", "hijos_id"),
        "interest_vocab": get_vocab_size(dataset, "user", "interest_id"),
        "marital_vocab": get_vocab_size(dataset, "user", "marital_status_id"),
        "personality_vocab": get_vocab_size(dataset, "user", "personality_id"),
        "religion_vocab": get_vocab_size(dataset, "user", "religion_id"),
        "transport_vocab": get_vocab_size(dataset, "user", "transport_id"),
    },

    "place_tower": {
        "numeric_dim": 12,
        "ordinal_dim": 3,
        "cuisine_vocab_size": NUM_CUISINES,
        "bert_dim": BERT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "smoking_vocab": get_vocab_size(dataset, "place", "smoking_area_id"),
        "rambience_vocab": get_vocab_size(dataset, "place", "rambience_id"),
        "parking_vocab": get_vocab_size(dataset, "place", "parking_lot_id"),
    }
}



user_tower =UserTower(**model_config["user_tower"]).to(device)

place_tower = PlaceTower(**model_config["place_tower"]).to(device)

torch.save({
    "user_tower_state": user_tower.state_dict(),
    "place_tower_state": place_tower.state_dict(),
    "config": model_config
}, "models/twotower/twotower_model.pt")

model = TwoTowerModel(user_tower, place_tower)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)



TwoTowerModel(
  (user_tower): UserTower(
    (dress_emb): Embedding(4, 8)
    (ambience_emb): Embedding(3, 8)
    (transport_emb): Embedding(3, 8)
    (marital_emb): Embedding(3, 4)
    (hijos_emb): Embedding(3, 4)
    (interest_emb): Embedding(5, 8)
    (personality_emb): Embedding(4, 8)
    (religion_emb): Embedding(5, 4)
    (activity_emb): Embedding(4, 8)
    (cuisine_emb): Embedding(103, 32)
    (mlp): Sequential(
      (0): Linear(in_features=1634, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (place_tower): PlaceTower(
    (smoking_emb): Embedding(5, 4)
    (rambience_emb): Embedding(2, 8)
    (parking_emb): Embedding(4, 4)
    (cuisine_emb): Embedding(23, 32)
    (mlp): Sequential(
      (0): Linear(in_features=1599, out_features=128, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=128, bias=

In [7]:
print(device)

cuda


In [8]:
def recall_at_k(user_emb, place_emb, k=10):

    sim = torch.matmul(user_emb, place_emb.T)

    topk = sim.topk(k, dim=1).indices

    targets = torch.arange(user_emb.size(0), device=user_emb.device)

    hits = (topk == targets.unsqueeze(1)).any(dim=1).float()

    return hits.mean().item()

In [9]:
from tqdm.notebook import tqdm

def train_model(model, train_loader, val_loader, epochs=200, lr=1e-4, device="cuda"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = ContrastiveLoss(temperature=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_recall = 0.0
    patience = 5
    no_improve = 0
    model.to(device)
    

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        # Wrap the loader with tqdm for a progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch")
        
        for batch in pbar:
            # Move batch to device
            user_feat = {k: v.to(device) for k, v in batch["user"].items()}
            place_feat = {k: v.to(device) for k, v in batch["place"].items()}
            labels = batch["label"].to(device)
            
            # print(user_feat, place_feat)
            optimizer.zero_grad()
            
            # Inside the batch loop
            u_emb, p_emb = model(user_feat, place_feat)
            

            # if torch.isnan(u_emb).any() or torch.isnan(p_emb).any():
            #     print("Found NaN in embeddings!")
            #     # Check if inputs have NaNs
            #     for feat in u_emb:
            #         if torch.isnan(feat).any():
            #             print(f"!!! NaN detected in {feat} features !!!")
            #     for feat in p_emb:
            #         if torch.isnan(feat).any():
            #             print(f"!!! NaN detected in {feat} features !!!")
            #     break

            loss = criterion(u_emb, p_emb, labels)

            if torch.isnan(loss):
                print("Loss became NaN!")
                print(f"Logits max: {torch.matmul(u_emb, p_emb.T).max()}")
                break
            
            # Backward pass
            loss.backward()
            
            # Optional: Gradient Clipping (uncomment if training is unstable)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            
            # Update progress bar stats
            
            batch_loss = loss.item()
            total_loss += batch_loss
            pbar.set_postfix({"batch_loss": f"{batch_loss:.4f}"})
            
        avg_loss = total_loss / len(train_loader)
        
        

        #Validation
        model.eval()
        val_loss = 0
        recallk=0 #default k is 10

        with torch.no_grad():

            pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", unit="batch")

            for batch in pbar:

                user_feat = {k: v.to(device) for k, v in batch["user"].items()}
                place_feat = {k: v.to(device) for k, v in batch["place"].items()}
                labels = batch["label"].to(device)

                u_emb, p_emb = model(user_feat, place_feat)

                loss = criterion(u_emb, p_emb, labels)

                batch_loss = loss.item()
                val_loss += batch_loss

                recallk += recall_at_k(u_emb, p_emb, k=10)

                pbar.set_postfix({"val_loss": f"{batch_loss:.4f}"})
        
        val_loss /= len(val_loader)
        recallk /= len(val_loader)

        scheduler.step(recallk)

        #early stopping
        if recallk > best_recall:
            best_recall = recallk
            no_improve=0
            # torch.save(model.state_dict(), "models/twotower/best.pt")
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch
            }, "models/twotower/best_checkpoint.pt")
        else:
            no_improve+=1
            if no_improve >= patience:
                print("early stopping")
                break
 
        print(f"✅ Epoch {epoch+1} Complete | Average Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")


In [10]:
train_model(model, train_loader, val_loader)

Epoch 1/200:   0%|          | 0/4 [00:00<?, ?batch/s]

Epoch 1/200 [Val]:   0%|          | 0/4 [00:00<?, ?batch/s]

✅ Epoch 1 Complete | Average Loss: 5.5213 | Val Loss: 5.4451


e:\Recomm2.0\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 2/200:   0%|          | 0/4 [00:00<?, ?batch/s]

...

Epoch 125/200 [Val]:   0%|          | 0/4 [00:00<?, ?batch/s]

✅ Epoch 125 Complete | Average Loss: 4.0256 | Val Loss: 3.5578


Epoch 126/200:   0%|          | 0/4 [00:00<?, ?batch/s]

Epoch 126/200 [Val]:   0%|          | 0/4 [00:00<?, ?batch/s]

early stopping


In [11]:
dataset[0]

{'user': {'height_norm': tensor(0.3266),
  'weight_norm': tensor(0.0076),
  'age_norm': tensor(-0.1573),
  'budget_ord': tensor(0.3333),
  'drink_level_ord': tensor(0.3333),
  'is_smoker': tensor(0.),
  'bert_emb': array([ 0.05885682, -0.0321352 ,  0.01479898, ...,  0.07019871,
          0.04484327, -0.03281486], shape=(1536,), dtype=float32),
  'cuisine_ids': [67],
  'dress_preference_id': 0,
  'ambience_id': 0,
  'transport_id': 2,
  'marital_status_id': 0,
  'hijos_id': 2,
  'interest_id': 3,
  'personality_id': 3,
  'religion_id': 0,
  'activity_id': 1},
 'place': {'price_ord': tensor(0.3333),
  'alcohol_ord': tensor(0.),
  'is_open': tensor(0.),
  'rating_bayes': tensor(0.2609),
  'food_bayes': tensor(0.2833),
  'service_bayes': tensor(0.2300),
  'rating_min': tensor(0.),
  'food_min': tensor(0.),
  'service_min': tensor(0.),
  'rating_max': tensor(0.4000),
  'food_max': tensor(0.4000),
  'service_max': tensor(0.4000),
  'rating_count': tensor(3.6109),
  'food_count': tensor(3.610

In [12]:
def train_with_val(checkpoint, loader, epochs=40, lr=1e-4, device="cuda"):

    model.load_state_dict(checkpoint["model"])
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scheduler.load_state_dict(checkpoint["scheduler"])
    criterion = ContrastiveLoss(temperature=0.1)
    
    model.train()

    for epoch in range(epochs):

        total_loss = 0
        pbar = tqdm(loader, desc=f"Final Epoch {epoch+1}/{epochs}")

        for batch in pbar:

            user_feat = {k: v.to(device) for k, v in batch["user"].items()}
            place_feat = {k: v.to(device) for k, v in batch["place"].items()}
            labels = batch["label"].to(device)

            optimizer.zero_grad()

            u_emb, p_emb = model(user_feat, place_feat)

            loss = criterion(u_emb, p_emb, labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            optimizer.step()

            total_loss += loss.item()

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()

        print(f"Epoch {epoch+1} | Loss {total_loss/len(loader):.4f}")
    
    model.eval()
    torch.save({ "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": epoch }, "models/twotower/best_checkpoint_with_val.pt")
    torch.save({ "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": epoch }, "models/twotower/best_checkpoint_runtime.pt")

In [13]:
checkpoint = torch.load("models/twotower/best_checkpoint.pt", map_location=device)
train_with_val(checkpoint, val_loader)

C:\Users\parac\AppData\Local\Temp\ipykernel_32504\2409967293.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("models/twotower/best_checkpoint.pt"

Final Epoch 1/40:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 1 | Loss 4.0030


Final Epoch 2/40:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 2 | Loss 4.0179


...

Epoch 38 | Loss 3.8035


Final Epoch 39/40:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 39 | Loss 3.8104


Final Epoch 40/40:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 40 | Loss 3.8304


In [14]:
# Now, with this model, need to save the place and user tower separately
# Calculate and store the place embeddings user embeddings in db

In [15]:
import psycopg2.extras as extras
import psycopg2

@torch.no_grad()
def generate_and_save_embeddings(checkpoint, user_loader, place_loader, conn, device="cuda"):
    model.load_state_dict(checkpoint["model"])
    model.to(device)
    model.eval()
    
    # --- 1. Process Users ---
    user_updates = []
    for batch in tqdm(user_loader, desc="User Embeddings"):
        db_user_ids = batch["ids"] 
        user_features = {k: v.to(device) for k, v in batch["features"].items()}
        u_emb = model.user_tower(user_features)
        vectors = u_emb.cpu().numpy()
        
        for db_id, vector in zip(db_user_ids, vectors):
            user_updates.append((vector.tolist(), db_id))
    
    # Update users in one go
    batch_update(conn, "user_profiles", "userID", user_updates)

    # --- 2. Process Places ---
    place_updates = []
    for batch in tqdm(place_loader, desc="Place Embeddings"):
        db_place_ids = batch["ids"]
        place_features = {k: v.to(device) for k, v in batch["features"].items()}
        p_emb = model.place_tower(place_features)
        vectors = p_emb.cpu().numpy()
        
        for db_id, vector in zip(db_place_ids, vectors):
            place_updates.append((vector.tolist(), db_id))
            
    batch_update(conn, "places", "placeID", place_updates)

def batch_update(conn, table, id_col, data):
    query = f"""
        UPDATE {table} SET nn_embedding = v.emb
        FROM (VALUES %s) AS v(emb, id)
        WHERE {table}.{id_col} = v.id
    """
    with conn.cursor() as cur:
        # template=None tells extras to figure out the types automatically
        extras.execute_values(cur, query, data, template=None, page_size=100)
    conn.commit()

In [16]:
# Create connection for these loaders
conn_gen = psycopg2.connect(database='reco_db', user='reco_user', password='reco_pass')
from tqdm.notebook import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
user_emb_loader = DataLoader(
    UserEmbeddingDataset(conn_gen),
    batch_size=256,
    shuffle=False,
    collate_fn=user_collate_fn
)

place_emb_loader = DataLoader(
    PlaceEmbeddingDataset(conn_gen),
    batch_size=256,
    shuffle=False,
    collate_fn=place_collate_fn
)

checkpoint = torch.load("models/twotower/best_checkpoint_with_val.pt", map_location=device)
generate_and_save_embeddings(checkpoint, user_emb_loader, place_emb_loader, conn_gen)

C:\Users\parac\AppData\Local\Temp\ipykernel_32504\3096671413.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("models/twotower/best_checkpoint_wi

User Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Place Embeddings:   0%|          | 0/1 [00:00<?, ?it/s]